# BanglaLLM 7B Instruct — Zero-Shot Government QA Evaluation

This notebook is the **BanglaLLM version** of the supplied Qwen zero-shot notebook.

Model:

`BanglaLLM/bangla-llama-7b-instruct-v0.1`

Test data:

`/kaggle/input/datasets/akra1234/government/merged_test_data.csv`

The training file is intentionally **not used** for prompting, fine-tuning, or demonstrations.

## What stays the same
- Same test dataset
- Same true zero-shot evaluation setup
- Same normalization and row-level metrics
- Same BLEU, ROUGE, METEOR and multilingual BERTScore evaluation
- Same `prediction.csv` and `result.csv` outputs
- Same 4-bit NF4 loading style as the supplied Qwen notebook

## BanglaLLM-specific changes
- Replaces `Qwen/Qwen2.5-7B-Instruct` with `BanglaLLM/bangla-llama-7b-instruct-v0.1`.
- Uses BanglaLLM's `### Instruction / ### Input / ### Response` prompt format instead of Qwen's chat template.
- Uses the checkpoint's sampling-style decoding: `do_sample=True`, `temperature=0.6`, `top_p=0.9`.
- Uses a fixed 256-token generation safety cap; it does **not** automatically expand 256 → 512 → 1024 → 2048 → 4096.
- Records whether an answer reached the token cap, but still evaluates all model outputs. A failure to terminate is itself part of zero-shot model behavior.

Outputs:
- `/kaggle/working/prediction.csv`
- `/kaggle/working/result.csv`

Metrics:
- Normalized Exact Match
- Token F1
- Fuzzy Match
- Corpus BLEU
- ROUGE-1
- ROUGE-2
- ROUGE-L
- METEOR
- BERTScore Precision
- BERTScore Recall
- BERTScore F1
- Token-limit Outputs


In [1]:
# ============================================================
# CELL 1 — INSTALL DEPENDENCIES
# ============================================================

!pip install -q -U "transformers>=4.45,<5" accelerate bitsandbytes sentencepiece \
    sacrebleu rapidfuzz nltk "bert-score==0.3.13"


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.7/11.7 MB 86.3 MB/s eta 0:00:00:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.2/389.2 kB 28.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 MB 39.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 87.0 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 72.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 516.0/516.0 kB 34.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.9/65.9 kB 3.7 MB/s eta 0:00:00


In [2]:
# ============================================================
# CELL 2 — IMPORTS + PATHS + LOAD TEST DATA
# ============================================================

import os
import re
import gc
import random
import unicodedata
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import torch

from tqdm.auto import tqdm
from rapidfuzz import fuzz
from sacrebleu.metrics import BLEU
from nltk.translate.meteor_score import meteor_score
from bert_score import score as bert_score

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    set_seed,
)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

MODEL_NAME = "BanglaLLM/bangla-llama-7b-instruct-v0.1"

# True zero-shot: this training file is intentionally NOT loaded or used.
TRAIN_JSONL = "/kaggle/input/datasets/akra1234/government/government_chat_train.jsonl"

TEST_CSV = "/kaggle/input/datasets/akra1234/government/merged_test_data.csv"

OUT_DIR = Path("/kaggle/working")
PRED_PATH = OUT_DIR / "prediction.csv"
PARTIAL_PATH = OUT_DIR / "prediction_partial.csv"
RESULT_PATH = OUT_DIR / "result.csv"

assert os.path.exists(TEST_CSV), f"Test file not found: {TEST_CSV}"

raw_df = pd.read_csv(TEST_CSV).fillna("")

print("Rows:", len(raw_df))
print("Columns:", raw_df.columns.tolist())


def choose_column(columns, candidates, required=True):
    lookup = {str(c).lower(): c for c in columns}
    for name in candidates:
        if name.lower() in lookup:
            return lookup[name.lower()]
    if required:
        raise ValueError(
            f"Could not find any of {candidates}. Available columns: {list(columns)}"
        )
    return None


QUESTION_COL = choose_column(
    raw_df.columns,
    ["instruction", "question", "prompt", "query"]
)

GOLD_COL = choose_column(
    raw_df.columns,
    ["output", "gold", "reference", "answer", "target"]
)

INPUT_COL = choose_column(
    raw_df.columns,
    ["input"],
    required=False
)

print("Question column:", QUESTION_COL)
print("Gold column:", GOLD_COL)
print("Optional input column:", INPUT_COL)

df = raw_df.copy()

df["question"] = df[QUESTION_COL].astype(str).str.strip()
df["gold"] = df[GOLD_COL].astype(str).str.strip()

if INPUT_COL is not None and INPUT_COL != QUESTION_COL:
    df["extra_input"] = df[INPUT_COL].astype(str).str.strip()
else:
    df["extra_input"] = ""

# Sampling is used for this checkpoint, so initialize Transformers' RNG too.
set_seed(SEED)

display(df.head(3))


Rows: 248
Columns: ['id', 'domain', 'topic', 'question_type', 'instruction', 'input', 'output', 'source_url', 'split', 'source']
Question column: instruction
Gold column: output
Optional input column: input


,id,domain,topic,question_type,instruction,input,output,source_url,split,source,question,gold,model_input
0,nid_003,nid,nid_number_structure,documents,NID আবেদন করতে কী কী ডকুমেন্ট লাগে?,,"প্রিন্টেড আবেদনপত্র, পাসপোর্ট সাইজ ছবি, ১৭ ডিজ...",https://services.nidw.gov.bd/,test,NID,NID আবেদন করতে কী কী ডকুমেন্ট লাগে?,"প্রিন্টেড আবেদনপত্র, পাসপোর্ট সাইজ ছবি, ১৭ ডিজ...",NID আবেদন করতে কী কী ডকুমেন্ট লাগে?
1,nid_006,nid,eligibility,procedure,NID কারা আবেদন করতে পারবে?,,০১ অক্টোবর ২০১০ এর আগে জন্মগ্রহণকারী বাংলাদেশী...,https://services.nidw.gov.bd/,test,NID,NID কারা আবেদন করতে পারবে?,০১ অক্টোবর ২০১০ এর আগে জন্মগ্রহণকারী বাংলাদেশী...,NID কারা আবেদন করতে পারবে?
2,nid_007,nid,new_voter_registration,procedure,যদি আগে ভোটার হয়ে থাকি তাহলে কি আবার আবেদন করত...,,"না, আগে ভোটার হয়ে থাকলে নতুন নিবন্ধনের প্রয়োজন...",https://services.nidw.gov.bd/,test,NID,যদি আগে ভোটার হয়ে থাকি তাহলে কি আবার আবেদন করত...,"না, আগে ভোটার হয়ে থাকলে নতুন নিবন্ধনের প্রয়োজন...",যদি আগে ভোটার হয়ে থাকি তাহলে কি আবার আবেদন করত...


In [3]:
# ============================================================
# CELL 3 — LOAD BANGLALLM 7B INSTRUCT IN 4-BIT
# ============================================================

if not torch.cuda.is_available():
    raise RuntimeError(
        "GPU is strongly recommended. In Kaggle: Settings -> Accelerator -> GPU."
    )

compute_dtype = torch.float16

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=compute_dtype,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    use_fast=True,
)

if tokenizer.pad_token_id is None:
    if tokenizer.eos_token_id is None:
        raise ValueError("Tokenizer has neither PAD nor EOS token configured.")
    tokenizer.pad_token = tokenizer.eos_token

tokenizer.padding_side = "left"

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=compute_dtype,
)

model.eval()

MODEL_CONTEXT_WINDOW = int(
    getattr(model.config, "max_position_embeddings", 4096)
)

print("Loaded:", MODEL_NAME)
print("Device:", next(model.parameters()).device)
print("Context window:", MODEL_CONTEXT_WINDOW)
print("Tokenizer EOS:", tokenizer.eos_token_id)
print("Model generation EOS:", model.generation_config.eos_token_id)


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

Loaded: Qwen/Qwen2.5-7B-Instruct
Device: cuda:0


In [4]:
# ============================================================
# CELL 4 — TRUE ZERO-SHOT BANGLALLM GENERATION
# ============================================================

SYSTEM_PROMPT = (
    "আপনি বাংলাদেশের সরকারি সেবা সম্পর্কিত প্রশ্নের সহায়ক। "
    "ব্যবহারকারীর প্রশ্নের উত্তর বাংলায় দিন। "
    "উত্তরটি সংক্ষিপ্ত, সরাসরি ও তথ্যভিত্তিক রাখুন। "
    "তথ্য না জানলে বানিয়ে বলবেন না। "
    "কোনো reference answer, dataset, training example বা evaluation-এর কথা উল্লেখ করবেন না।"
)

BATCH_SIZE = 4

# BanglaLLM/LLaMA has a 4096-token context window.
MAX_INPUT_TOKENS = 1536
MAX_NEW_TOKENS = 256

# BanglaLLM generation settings.
DO_SAMPLE = True
TEMPERATURE = 0.6
TOP_P = 0.9
TOP_K = 50

# True = delete stale Kaggle working outputs and regenerate every row.
FORCE_REGENERATE = True


def make_prompt(instruction, extra_input=""):
    """
    BanglaLLM instruction format.

    Without extra input:
        system
        ### Instruction:
        ...
        ### Response:

    With extra input:
        system
        ### Instruction:
        ...
        ### Input:
        ...
        ### Response:
    """
    instruction = str(instruction).strip()
    extra_input = str(extra_input).strip()

    if extra_input:
        return (
            f"{SYSTEM_PROMPT}\n\n"
            f"### Instruction:\n{instruction}\n\n"
            f"### Input:\n{extra_input}\n\n"
            "### Response:\n"
        )

    return (
        f"{SYSTEM_PROMPT}\n\n"
        f"### Instruction:\n{instruction}\n\n"
        "### Response:\n"
    )


def _get_eos_token_ids():
    eos = model.generation_config.eos_token_id

    if eos is None:
        eos = tokenizer.eos_token_id

    if eos is None:
        raise ValueError("No EOS token id is configured.")

    eos_ids = [eos] if isinstance(eos, int) else list(eos)

    if tokenizer.eos_token_id is not None:
        eos_ids.append(int(tokenizer.eos_token_id))

    return list(dict.fromkeys(int(x) for x in eos_ids))


EOS_TOKEN_IDS = _get_eos_token_ids()
GEN_EOS = EOS_TOKEN_IDS[0] if len(EOS_TOKEN_IDS) == 1 else EOS_TOKEN_IDS
EOS_TOKEN_ID_SET = set(EOS_TOKEN_IDS)

if MAX_INPUT_TOKENS + MAX_NEW_TOKENS > MODEL_CONTEXT_WINDOW:
    raise ValueError(
        f"Token budget exceeds context window: "
        f"{MAX_INPUT_TOKENS} + {MAX_NEW_TOKENS} > {MODEL_CONTEXT_WINDOW}"
    )

print("EOS token ids:", EOS_TOKEN_IDS)
print("Model context window:", MODEL_CONTEXT_WINDOW)
print("Max prompt tokens:", MAX_INPUT_TOKENS)
print("Max new tokens:", MAX_NEW_TOKENS)
print("Sampling:", DO_SAMPLE)
print("temperature/top_p/top_k:", TEMPERATURE, TOP_P, TOP_K)


def generate_batch(instructions, extra_inputs):
    prompts = [
        make_prompt(inst, extra)
        for inst, extra in zip(instructions, extra_inputs)
    ]

    batch = tokenizer(
        prompts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=MAX_INPUT_TOKENS,
    )

    input_device = next(model.parameters()).device
    batch = {k: v.to(input_device) for k, v in batch.items()}

    input_width = batch["input_ids"].shape[1]

    with torch.inference_mode():
        generated = model.generate(
            **batch,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=DO_SAMPLE,
            temperature=TEMPERATURE,
            top_p=TOP_P,
            top_k=TOP_K,
            num_beams=1,
            use_cache=True,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=GEN_EOS,
        )

    new_tokens = generated[:, input_width:]
    token_rows = new_tokens.detach().cpu().tolist()

    answers = []
    generated_lengths = []
    eos_flags = []
    token_limit_flags = []

    for token_ids in token_rows:
        first_eos_position = next(
            (
                j
                for j, tok in enumerate(token_ids)
                if tok in EOS_TOKEN_ID_SET
            ),
            None,
        )

        if first_eos_position is not None:
            effective_ids = token_ids[:first_eos_position + 1]
            generated_length = first_eos_position + 1
            finished_with_eos = True
            hit_token_limit = False
        else:
            effective_ids = token_ids[:MAX_NEW_TOKENS]
            generated_length = min(len(token_ids), MAX_NEW_TOKENS)
            finished_with_eos = False
            hit_token_limit = generated_length >= MAX_NEW_TOKENS

        answer = tokenizer.decode(
            effective_ids,
            skip_special_tokens=True,
            clean_up_tokenization_spaces=False,
        ).strip()

        answers.append(answer)
        generated_lengths.append(int(generated_length))
        eos_flags.append(bool(finished_with_eos))
        token_limit_flags.append(bool(hit_token_limit))

    return (
        answers,
        generated_lengths,
        eos_flags,
        token_limit_flags,
    )


def save_partial(frame):
    frame.sort_values("row_index").to_csv(
        PARTIAL_PATH,
        index=False,
        encoding="utf-8-sig",
    )


# ------------------------------------------------------------
# Clean start
# ------------------------------------------------------------

if FORCE_REGENERATE:
    for old_path in [PARTIAL_PATH, PRED_PATH, RESULT_PATH]:
        if old_path.exists():
            old_path.unlink()
            print("Removed old output:", old_path)

predictions = []

if (not FORCE_REGENERATE) and PARTIAL_PATH.exists():
    partial = pd.read_csv(PARTIAL_PATH).fillna("")

    required_cols = {
        "row_index",
        "question",
        "gold",
        "prediction",
        "generated_tokens",
        "finished_with_eos",
        "hit_token_limit",
        "truncated",
        "max_new_tokens_used",
        "generation_attempts",
    }

    expected_indices = list(range(len(partial)))
    actual_indices = pd.to_numeric(
        partial.get("row_index", pd.Series(dtype=int)),
        errors="coerce",
    ).tolist()

    if (
        len(partial) <= len(df)
        and required_cols.issubset(partial.columns)
        and actual_indices == expected_indices
    ):
        predictions = partial.to_dict("records")
        print(f"Resuming from {len(predictions)} completed rows.")
    else:
        print("Ignoring incompatible partial file and starting fresh.")


# ------------------------------------------------------------
# Generate all test rows once
# ------------------------------------------------------------

start = len(predictions)

for i in tqdm(
    range(start, len(df), BATCH_SIZE),
    desc=f"BanglaLLM zero-shot generation ({MAX_NEW_TOKENS} max tokens)",
):
    end = min(i + BATCH_SIZE, len(df))

    questions = df.iloc[i:end]["question"].tolist()
    extra_inputs = df.iloc[i:end]["extra_input"].tolist()

    # Reproducible sampling at batch level.
    set_seed(SEED + i)

    (
        answers,
        generated_lengths,
        eos_flags,
        token_limit_flags,
    ) = generate_batch(
        questions,
        extra_inputs,
    )

    for local_idx, (
        answer,
        gen_len,
        eos_ok,
        hit_limit,
    ) in enumerate(
        zip(
            answers,
            generated_lengths,
            eos_flags,
            token_limit_flags,
        )
    ):
        row_idx = i + local_idx
        row = df.iloc[row_idx]

        record = {
            "row_index": row_idx,
            "question": row["question"],
            "gold": row["gold"],
            "prediction": answer,
            "generated_tokens": int(gen_len),
            "finished_with_eos": bool(eos_ok),
            "hit_token_limit": bool(hit_limit),

            # Kept for compatibility with the supplied Qwen notebook.
            # Here "truncated" means the model reached MAX_NEW_TOKENS
            # without emitting EOS.
            "truncated": bool(hit_limit),

            "max_new_tokens_used": int(MAX_NEW_TOKENS),
            "generation_attempts": 1,
        }

        for col in [
            "id",
            "domain",
            "topic",
            "question_type",
            "source_url",
            "split",
        ]:
            if col in df.columns:
                record[col] = row[col]

        predictions.append(record)

    save_partial(pd.DataFrame(predictions))


pred_df = (
    pd.DataFrame(predictions)
    .sort_values("row_index")
    .reset_index(drop=True)
)

assert len(pred_df) == len(df), (
    f"Generated {len(pred_df)} predictions for {len(df)} test rows."
)

pred_df.to_csv(
    PRED_PATH,
    index=False,
    encoding="utf-8-sig",
)

token_limit_count = int(
    pred_df["hit_token_limit"].astype(bool).sum()
)

empty_output_count = int(
    pred_df["prediction"].astype(str).str.strip().eq("").sum()
)

print("\nGeneration complete:", len(pred_df))
print("Hit token limit:", token_limit_count)
print("Empty outputs:", empty_output_count)
print("Saved predictions:", PRED_PATH)

# IMPORTANT:
# We do not automatically retry token-limit outputs with larger budgets.
# A model that fails to terminate is itself an observable zero-shot behavior.
# The predictions are still evaluated, and the token-limit count is reported.
display(pred_df.head(3))


Generation EOS token ids: [151645, 151643]
Generation budgets: (512, 1024, 2048, 4096)


Qwen zero-shot generation (512 tokens):   0%|          | 0/62 [00:00<?, ?it/s]

After 512-token pass, truncated outputs: 38
Retrying 38 truncated outputs with max_new_tokens=1024...


Retry at 1024 tokens:   0%|          | 0/38 [00:00<?, ?it/s]

Remaining truncated after 1024-token retry: 5
Retrying 5 truncated outputs with max_new_tokens=2048...


Retry at 2048 tokens:   0%|          | 0/5 [00:00<?, ?it/s]

Remaining truncated after 2048-token retry: 0
Generation complete: 248
Truncated outputs: 0
Saved fresh predictions: /kaggle/working/prediction.csv
Maximum generation budget actually used: 2048


,row_index,question,gold,prediction,truncated,generated_tokens,max_new_tokens_used,generation_attempts,id,domain,topic,question_type,source_url,split
0,0,NID আবেদন করতে কী কী ডকুমেন্ট লাগে?,"প্রিন্টেড আবেদনপত্র, পাসপোর্ট সাইজ ছবি, ১৭ ডিজ...",NID আবেদন করতে প্রয়োজনীয় ডকুমেন্টগুলি অন্যান...,False,717,1024,2,nid_003,nid,nid_number_structure,documents,https://services.nidw.gov.bd/,test
1,1,NID কারা আবেদন করতে পারবে?,০১ অক্টোবর ২০১০ এর আগে জন্মগ্রহণকারী বাংলাদেশী...,NID (National Identity Card) আবেদন করতে পারবেন...,False,503,512,1,nid_006,nid,eligibility,procedure,https://services.nidw.gov.bd/,test
2,2,যদি আগে ভোটার হয়ে থাকি তাহলে কি আবার আবেদন করত...,"না, আগে ভোটার হয়ে থাকলে নতুন নিবন্ধনের প্রয়োজন...","না, যদি আগে ভোটার হয়ে থাকেন তাহলে আবার আবেদন ...",False,258,512,1,nid_007,nid,new_voter_registration,procedure,https://services.nidw.gov.bd/,test


In [5]:
# ============================================================
# CELL 5 — FREE BANGLALLM GPU MEMORY BEFORE BERTSCORE
# ============================================================

del model

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("BanglaLLM model removed from memory.")


Qwen model removed from memory.


In [6]:
# ============================================================
# CELL 6 — METRIC FUNCTIONS
# Keeps the same normalization / Token F1 / ROUGE definitions
# as the previous evaluation pipeline.
# ============================================================

BN_TO_EN = str.maketrans(
    "০১২৩৪৫৬৭৮৯",
    "0123456789"
)


def normalize(text):

    text = unicodedata.normalize(
        "NFKC",
        str(text)
    )

    text = text.translate(
        BN_TO_EN
    ).lower()

    text = re.sub(
        r"[^\u0980-\u09FFA-Za-z0-9]+",
        " ",
        text
    )

    return re.sub(
        r"\s+",
        " ",
        text
    ).strip()


def tokens(text):
    return normalize(text).split()


# ---------------- Normalized Exact Match ----------------

def normalized_exact_match(pred, gold):

    return float(
        normalize(pred)
        ==
        normalize(gold)
    )


# ---------------- Token F1 ----------------

def token_f1(pred, gold):

    p = tokens(pred)
    g = tokens(gold)

    if not p and not g:
        return 1.0

    if not p or not g:
        return 0.0

    overlap = sum(
        (
            Counter(p)
            &
            Counter(g)
        ).values()
    )

    if overlap == 0:
        return 0.0

    precision = overlap / len(p)
    recall = overlap / len(g)

    return (
        2 * precision * recall
        /
        (precision + recall)
    )


# ---------------- ROUGE-N F1 ----------------

def rouge_n(pred, gold, n):

    p = tokens(pred)
    g = tokens(gold)

    if len(p) < n or len(g) < n:
        return 0.0

    pg = Counter(
        tuple(p[i:i+n])
        for i in range(len(p)-n+1)
    )

    gg = Counter(
        tuple(g[i:i+n])
        for i in range(len(g)-n+1)
    )

    overlap = sum(
        (pg & gg).values()
    )

    if overlap == 0:
        return 0.0

    precision = overlap / sum(pg.values())
    recall = overlap / sum(gg.values())

    return (
        2 * precision * recall
        /
        (precision + recall)
    )


# ---------------- ROUGE-L F1 ----------------

def rouge_l(pred, gold):

    p = tokens(pred)
    g = tokens(gold)

    if not p and not g:
        return 1.0

    if not p or not g:
        return 0.0

    dp = [0] * (len(g) + 1)

    for x in p:

        new = [0]

        for j, y in enumerate(g, 1):

            if x == y:
                new.append(dp[j-1] + 1)

            else:
                new.append(
                    max(dp[j], new[-1])
                )

        dp = new

    lcs = dp[-1]

    precision = lcs / len(p)
    recall = lcs / len(g)

    if precision + recall == 0:
        return 0.0

    return (
        2 * precision * recall
        /
        (precision + recall)
    )


# ---------------- Bengali-safe METEOR ----------------
# NLTK's default METEOR uses English Porter stemming + English WordNet.
# For Bangla evaluation, disable those English-only lexical resources while
# retaining METEOR's exact-token alignment and fragmentation penalty.

class IdentityStemmer:
    def stem(self, word):
        return word


class EmptyWordNet:
    def synsets(self, word):
        return []


IDENTITY_STEMMER = IdentityStemmer()
EMPTY_WORDNET = EmptyWordNet()


def meteor_bn(pred, gold):

    p = tokens(pred)
    g = tokens(gold)

    if not p and not g:
        return 1.0

    if not p or not g:
        return 0.0

    return meteor_score(
        [g],
        p,
        stemmer=IDENTITY_STEMMER,
        wordnet=EMPTY_WORDNET,
    )


In [7]:
# ============================================================
# CELL 7 — COMPUTE ROW-LEVEL METRICS
# ============================================================

eval_df = pd.read_csv(PRED_PATH).fillna("")

# For a zero-shot benchmark, all model outputs are evaluated.
# If BanglaLLM reaches the generation cap, that failure is retained rather
# than silently excluding the row or giving it a larger budget.
n_token_limit_for_eval = 0
if "hit_token_limit" in eval_df.columns:
    n_token_limit_for_eval = int(
        eval_df["hit_token_limit"]
        .astype(str)
        .str.lower()
        .eq("true")
        .sum()
    )

print(f"Evaluating all {len(eval_df)} zero-shot predictions...")
print("Token-limit outputs included in evaluation:", n_token_limit_for_eval)

eval_df["Normalized Exact Match"] = [
    normalized_exact_match(p, g)
    for p, g in zip(
        eval_df["prediction"],
        eval_df["gold"]
    )
]

eval_df["Token F1"] = [
    token_f1(p, g)
    for p, g in zip(
        eval_df["prediction"],
        eval_df["gold"]
    )
]

eval_df["Fuzzy Match"] = [
    fuzz.token_set_ratio(
        normalize(p),
        normalize(g)
    ) / 100
    for p, g in zip(
        eval_df["prediction"],
        eval_df["gold"]
    )
]

eval_df["ROUGE-1"] = [
    rouge_n(p, g, 1)
    for p, g in zip(
        eval_df["prediction"],
        eval_df["gold"]
    )
]

eval_df["ROUGE-2"] = [
    rouge_n(p, g, 2)
    for p, g in zip(
        eval_df["prediction"],
        eval_df["gold"]
    )
]

eval_df["ROUGE-L"] = [
    rouge_l(p, g)
    for p, g in zip(
        eval_df["prediction"],
        eval_df["gold"]
    )
]

eval_df["METEOR"] = [
    meteor_bn(p, g)
    for p, g in tqdm(
        zip(
            eval_df["prediction"],
            eval_df["gold"]
        ),
        total=len(eval_df),
        desc="METEOR"
    )
]


# ============================================================
# CORPUS BLEU
# ============================================================

bleu = BLEU(
    tokenize="none",
    smooth_method="exp",
    effective_order=True
)

pred_texts = [
    " ".join(tokens(x))
    for x in eval_df["prediction"]
]

gold_texts = [
    " ".join(tokens(x))
    for x in eval_df["gold"]
]

corpus_bleu = (
    bleu.corpus_score(
        pred_texts,
        [gold_texts]
    ).score
    / 100
)

print("Corpus BLEU:", corpus_bleu)


Evaluating 248 fresh, non-truncated predictions...


METEOR:   0%|          | 0/248 [00:00<?, ?it/s]

Corpus BLEU: 0.01599351151156911


In [8]:
# ============================================================
# CELL 8 — BERTSCORE
# model: bert-base-multilingual-cased
# ============================================================

print("Calculating multilingual BERTScore...")

bert_device = "cuda" if torch.cuda.is_available() else "cpu"
bert_batch_size = 8 if torch.cuda.is_available() else 4

P, R, F1 = bert_score(
    eval_df["prediction"].astype(str).tolist(),
    eval_df["gold"].astype(str).tolist(),
    model_type="bert-base-multilingual-cased",
    batch_size=bert_batch_size,
    device=bert_device,
    idf=False,
    rescale_with_baseline=False,
    verbose=True
)

eval_df["BERTScore Precision"] = P.cpu().numpy()
eval_df["BERTScore Recall"] = R.cpu().numpy()
eval_df["BERTScore F1"] = F1.cpu().numpy()

print("BERTScore complete.")


Calculating multilingual BERTScore...


config.json:   0%|          | 0.00/625 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

calculating scores...
computing bert embedding.


  0%|          | 0/52 [00:00<?, ?it/s]

computing greedy matching.


  0%|          | 0/31 [00:00<?, ?it/s]

done in 2.74 seconds, 90.38 sentences/sec
BERTScore complete.


In [9]:
# ============================================================
# CELL 9 — FINAL RESULT + SAVE prediction.csv AND result.csv
# ============================================================

token_limit_count = int(
    eval_df["hit_token_limit"]
    .astype(str)
    .str.lower()
    .eq("true")
    .sum()
)

empty_output_count = int(
    eval_df["prediction"].astype(str).str.strip().eq("").sum()
)

avg_generated_tokens = (
    pd.to_numeric(eval_df["generated_tokens"], errors="coerce").mean()
    if "generated_tokens" in eval_df.columns else np.nan
)

max_generated_tokens = (
    pd.to_numeric(eval_df["generated_tokens"], errors="coerce").max()
    if "generated_tokens" in eval_df.columns else np.nan
)

result = pd.DataFrame({
    "metric": [
        "Normalized Exact Match",
        "Token F1",
        "Fuzzy Match",
        "Corpus BLEU",
        "ROUGE-1",
        "ROUGE-2",
        "ROUGE-L",
        "METEOR",
        "BERTScore Precision",
        "BERTScore Recall",
        "BERTScore F1",
        "Token-limit Outputs",
        "Empty Outputs",
        "Average Generated Tokens",
        "Maximum Generated Tokens",
    ],
    "score": [
        eval_df["Normalized Exact Match"].mean(),
        eval_df["Token F1"].mean(),
        eval_df["Fuzzy Match"].mean(),
        corpus_bleu,
        eval_df["ROUGE-1"].mean(),
        eval_df["ROUGE-2"].mean(),
        eval_df["ROUGE-L"].mean(),
        eval_df["METEOR"].mean(),
        eval_df["BERTScore Precision"].mean(),
        eval_df["BERTScore Recall"].mean(),
        eval_df["BERTScore F1"].mean(),
        token_limit_count,
        empty_output_count,
        avg_generated_tokens,
        max_generated_tokens,
    ]
})

# Save row-level predictions + all row-level metrics.
eval_df.to_csv(
    PRED_PATH,
    index=False,
    encoding="utf-8-sig"
)

# Save aggregate metrics.
result.to_csv(
    RESULT_PATH,
    index=False,
    encoding="utf-8-sig"
)

display(result)

print("\nSaved:")
print(PRED_PATH)
print(RESULT_PATH)

print("\nValidation:")
print("Rows evaluated:", len(eval_df))
print("Token-limit outputs:", token_limit_count)
print("Empty outputs:", empty_output_count)

print("\nNOTE:")
print("/kaggle/input is read-only. Kaggle outputs must be written under /kaggle/working.")


,metric,score
0,Normalized Exact Match,0.000000
1,Token F1,0.167548
2,Fuzzy Match,0.497470
3,Corpus BLEU,0.015994
4,ROUGE-1,0.167548
5,ROUGE-2,0.049497
6,ROUGE-L,0.138899
7,METEOR,0.170604
8,BERTScore Precision,0.671958
9,BERTScore Recall,0.718494



Saved:
/kaggle/working/prediction.csv
/kaggle/working/result.csv

Validation:
Rows evaluated: 248
Truncated outputs: 0
Empty outputs: 0

NOTE:
/kaggle/input is read-only. Kaggle outputs must be written under /kaggle/working.
